# Prune test (single run + post-hoc L1 pruning)

Single training run under fixed controls, then a post-hoc L1 pruning step applied to the `EQL.readout` weights.

**Controls**
- `stride_x = 5`, `stride_t = 5`
- `noise = 0.3`, `nu = 0.02`
- `steps = 8000`, `batch_size = 1000`, `lr = 0.001`
- `lam_pde = 0.5`, `lam_data = 10`, `lam_tv = 0`, `lam_reg = 0`

**Pruning**
- Threshold small-magnitude readout weights to zero (L1-style magnitude pruning).
- Report losses/coefficients before and after pruning.


In [ ]:
import sys

sys.path.append('..')  # add project root

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from prog import mlps, featlib, trainer, hlprs
import Datasets.matconv as mc

SEED = 1432
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cpu')


In [ ]:
# Controls

steps = 8000
log_every = 1000

noise = 0.3
nu = 0.02

stride_x = 5
stride_t = 5

lr = 1e-3
batch_size = 1000

lam_pde = 0.5
lam_data = 10.0
lam_tv = 0.0
lam_reg = 0.0

selected_derivs = ('u','u_x','u_xx')

# Dataset partitioning controls
part_num = 1
which_part = 1


In [ ]:
# Build dataset

partitions = mc.build_dataset_from_burgers(
    noise_level=noise,
    nu=nu,
    stride_t=stride_t,
    stride_x=stride_x,
    seed=SEED,
    quantile_splits=part_num,
    return_partitions=True,
)

key = [k for k in partitions if k.startswith(f'Q{which_part}:')][0]
t_np, x_np, y_np, y_noisy_np, _N = partitions[key]

# torch tensors

t_torch       = torch.from_numpy(t_np).to(device)
x_torch       = torch.from_numpy(x_np).to(device)
y_clean_torch = torch.from_numpy(y_np).to(device)
y_noisy_torch = torch.from_numpy(y_noisy_np).to(device)

print('N points:', t_np.shape[0])
print('x_unique:', len(np.unique(x_np)), 't_unique:', len(np.unique(t_np)))


In [ ]:
# Pruning trigger settings (trainer-controlled)

# Pruning is applied once when indicator norms fall below the enabled thresholds.
# Tune these to control WHEN pruning happens.

prune_enabled = True
prune_amount = 0.2

# Thresholds on the scalar norms returned by train.step(...):
# - coeff_mag: || |v params| ||_2
# - ema_grad:  || EMA(|grad|) ||_2
# - ema_drift: || EMA(|?param|) ||_2

prune_coeff_mag_max = None
prune_ema_grad_max = 1e-4
prune_ema_drift_max = 1e-5


In [ ]:
# Init models + trainer

u_model = mlps.SimpleMLP(n_layers=4, hidden_size=64, act=mlps.Sin)
symnet  = mlps.EQL(in_dim=len(selected_derivs), prod_dim=2, num_layers=1, bias=False)

train_config = trainer.TrainerConfig(
    lr=lr,
    lambda_pde=lam_pde,
    lambda_reg=lam_reg,
    lambda_tv=lam_tv,
    lambda_data=lam_data,
    selected_derivs=selected_derivs,
    device=device,
    prune_enabled=prune_enabled,
    prune_amount=prune_amount,
    prune_coeff_mag_max=prune_coeff_mag_max,
    prune_ema_grad_max=prune_ema_grad_max,
    prune_ema_drift_max=prune_ema_drift_max,
)

ft = featlib.FeatureTensor(selected_derivs, normalize=False)
feature_builder = ft.build

train = trainer.PDETrainer(
    u_model=u_model,
    v_model=symnet,
    cfg=train_config,
    feature_builder=feature_builder,
)


In [ ]:
# Train (single run)

loss_dat = []
loss_pde = []
loss_l1  = []
loss_tv  = []
loss_tot = []

coeff_mag_hist = []
ema_grad_hist = []
ema_drift_hist = []
prune_applied_hist = []
last_out = None

for i in range(steps):
    t, x, u_noisy, u_clean = hlprs.make_batch(
        batch_size=batch_size,
        t_torch=t_torch,
        x_torch=x_torch,
        y_clean=y_clean_torch,
        y_noisy=y_noisy_torch,
    )
    out = train.step(t=t, x=x, u_noisy=u_noisy, u_clean=u_clean)
    last_out = out

    loss_tot.append(out['loss'])
    loss_dat.append(out['loss_data'])
    loss_pde.append(out['loss_pde'])
    loss_l1.append(out['l1'])
    loss_tv.append(out['loss_tv'])

    coeff_mag_hist.append(out.get('coeff_mag', float('nan')))
    ema_grad_hist.append(out.get('ema_grad', float('nan')))
    ema_drift_hist.append(out.get('ema_drift', float('nan')))
    prune_applied_hist.append(out.get('prune_applied', 0))

    if log_every and i % log_every == 0:
        print(f"step {i} total={out['loss']:.6e} data={out['loss_data']:.6e} pde={out['loss_pde']:.6e} l1={out['l1']:.6e} coeff_mag={out.get('coeff_mag', float('nan')):.3e} ema_grad={out.get('ema_grad', float('nan')):.3e} ema_drift={out.get('ema_drift', float('nan')):.3e} pruned={out.get('prune_applied',0)}")

plt.figure(figsize=(7,4))
plt.plot(loss_tot, label='total')
plt.plot(loss_dat, label='data')
plt.plot(loss_pde, label='pde')
plt.plot(loss_l1, label='l1')
plt.yscale('log')
plt.xlabel('step')
plt.ylabel('loss')
plt.title('Training losses')
plt.legend()
plt.tight_layout()
plt.show()


print('prune_applied_final:', int(prune_applied_hist[-1]) if prune_applied_hist else 0)

plt.figure(figsize=(7,4))
plt.plot(coeff_mag_hist, label='coeff_mag')
plt.plot(ema_grad_hist, label='ema_grad')
plt.plot(ema_drift_hist, label='ema_drift')
plt.yscale('log')
plt.xlabel('step')
plt.ylabel('indicator (norm)')
plt.title('Pruning indicators')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Coefficients + snapshot after training (may include pruning)

print('readout.weight:')
print(symnet.readout.weight)

print('
effective quadratic matrix (unsym):')
print(symnet.effective_quadratic_matrix(symmetrize=False))

print('
M01, M10:')
M = symnet.effective_quadratic_matrix(symmetrize=False)
print('M01=', float(M[0,1].item()), 'M10=', float(M[1,0].item()))

hlprs.snapshot_comp(train.u, stride_x, stride_t, y_noisy_np, y_np, t_np, x_np, snap_no=10)[:0]


In [ ]:
# Evaluate losses on a fresh batch (no optimizer step)

mse = nn.MSELoss()

def eval_losses(trainer_obj, t, x, u_noisy):
    t = t.clone().detach().requires_grad_(True)
    x = x.clone().detach().requires_grad_(True)
    u_noisy = u_noisy.clone().detach()

    u_out = trainer_obj.u(t, x)
    loss_data = mse(u_out, u_noisy)

    features = trainer_obj.feature_builder(u_out, x=x)
    F = features.F
    u_t = torch.autograd.grad(u_out, t, grad_outputs=torch.ones_like(u_out), create_graph=False)[0]
    v_out = trainer_obj.v(F)
    loss_pde = mse(u_t, v_out)

    l1 = sum(p.abs().sum() for p in trainer_obj.v.parameters())

    return {
        'data_loss': float(loss_data.item()),
        'pde_loss': float(loss_pde.item()),
        'l1': float(l1.item()),
    }

# one evaluation batch
t_eval, x_eval, u_noisy_eval, _u_clean_eval = hlprs.make_batch(
    batch_size=batch_size,
    t_torch=t_torch,
    x_torch=x_torch,
    y_clean=y_clean_torch,
    y_noisy=y_noisy_torch,
)

before = eval_losses(train, t_eval, x_eval, u_noisy_eval)
print('before_prune:', before)


In [ ]:
# Inspect/remove pruning reparameterization (if pruning triggered)

import torch.nn.utils.prune as prune

print('trainer prune_applied:', getattr(train, '_prune_applied', None))
print('readout has weight_orig:', hasattr(symnet.readout, 'weight_orig'))
print('readout has weight_mask:', hasattr(symnet.readout, 'weight_mask'))

if hasattr(symnet.readout, 'weight_mask'):
    print('weight_mask:', symnet.readout.weight_mask.detach().cpu().numpy().astype(int).reshape(-1))

# Optional: make pruning permanent (fold mask into weight, remove reparam)
# if hasattr(symnet.readout, 'weight_orig'):
#     prune.remove(symnet.readout, 'weight')


## DeePyMoD Burgers scatter dataset export

This reproduces the dataset used for the scatterplot in `DeePyMoD-master/examples/PDE_Burgers.ipynb` and exports it as an `.npz`.


In [ ]:
# DEEPMOD_BURGERS_SCATTER_EXPORT
from pathlib import Path
import sys, json
import numpy as np
import torch

# Try import; fall back to local checkout path if needed
try:
    from deepymod.data import Dataset
    from deepymod.data.samples import Subsample_random
    from deepymod.data.burgers import burgers_delta
except ModuleNotFoundError:
    deepymod_root = Path(r"C:\\Users\\sami\\Documents\\DeePyMoD-master")
    if not deepymod_root.exists():
        raise FileNotFoundError(f"DeePyMoD not importable and not found at {deepymod_root}")
    sys.path.insert(0, str(deepymod_root))
    from deepymod.data import Dataset
    from deepymod.data.samples import Subsample_random
    from deepymod.data.burgers import burgers_delta

device = 'cuda' if torch.cuda.is_available() else 'cpu'
np.random.seed(42)
torch.manual_seed(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# Mirrors PDE_Burgers.ipynb dataset creation
v = 0.1
A = 1.0
x = torch.linspace(-3, 4, 100)
t = torch.linspace(0.5, 5.0, 50)
load_kwargs = {'x': x, 't': t, 'v': v, 'A': A}
preprocess_kwargs = {'noise_level': 0.05}

dataset = Dataset(
    burgers_delta,
    load_kwargs=load_kwargs,
    preprocess_kwargs=preprocess_kwargs,
    subsampler=Subsample_random,
    subsampler_kwargs={'number_of_samples': 2000},
    device=device,
)

coords = dataset.get_coords().detach().cpu().numpy()
data = dataset.get_data().detach().cpu().numpy()

export_path = (Path('..') / 'Datasets' / 'deepymod_burgers_scatter.npz').resolve()
meta = {
    'source': 'DeePyMoD-master/examples/PDE_Burgers.ipynb',
    'v': v,
    'A': A,
    'noise_level': preprocess_kwargs['noise_level'],
    'n_samples': int(coords.shape[0]),
    'coords_columns': ['t', 'x'],
    'data_columns': ['u'],
}
np.savez(export_path, coords=coords, data=data, meta=json.dumps(meta))
print('saved:', export_path)
print('coords shape:', coords.shape, 'data shape:', data.shape)
